# 04 - RAG pipeline 

Pipeline: pantry query -> alias map -> retrieve (overlap / BM25 / dense / hybrid) -> prompt -> local LLM (Ollama).

**Inputs** (made by notebooks 01 and 02): `data/processed/cleaned_recipes.parquet` (needs `canonical_ingredients`) and `data/processed/alias_map.json`.
`ingredient_utils.py` must be in `src/`.

Before running: Ollama must be running and `ollama pull llama3.2:3b` done.

In [2]:
%pip install -q ollama sentence-transformers pandas pyarrow numpy rank_bm25

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 0. Paths (works whether the notebook is run from `notebooks/` or the repo root)

In [3]:
import sys, os
from pathlib import Path

start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents) if (p / "src" / "ingredient_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Could not find src/ingredient_utils.py - open this notebook from inside NLP-Mini-Project.")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Working directory:", Path.cwd())

Working directory: C:\Users\micheala\Desktop\pinoy-recipe-rag\NLP-Mini-Project


## 1. Config

In [4]:
import re
import time
import numpy as np
import pandas as pd
import ollama
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from ingredient_utils import load_alias_map, build_lookup, normalize_text

LLM_MODEL = "llama3.2:3b"                       # exact Ollama tag
EMBED_MODEL = "intfloat/multilingual-e5-small"  # switch to -base later if time allows
TOP_K = 3                                       # recipes given to the LLM
RRF_K = 60                                      # Reciprocal Rank Fusion constant
SEED = 42
HYBRID_PARTS = ["bm25", "dense"]                # which retrievers the hybrid combines
GIVE_MATCH_HINTS = True                         # True: code computes matched/missing ingredients and gives them to the LLM

## 2. Load the cleaned recipes and the alias map

In [5]:
df = pd.read_parquet("data/processed/cleaned_recipes.parquet")
alias_map = load_alias_map("data/processed/alias_map.json")
lookup, collisions = build_lookup(alias_map)

for c in ["recipe_id", "recipe_name", "ingredients", "instructions", "canonical_ingredients"]:
    assert c in df.columns, f"Missing column {c}. Run notebooks 01 and 02 first. Available: {list(df.columns)}"

df["canon_list"] = df["canonical_ingredients"].map(list)
df["canon_set"] = df["canon_list"].map(set)

print(f"{len(df)} recipes | {len(alias_map)} canonical ingredients | {len(lookup)} lookup terms")
print("alias collisions:", collisions if collisions else "none")

1983 recipes | 189 canonical ingredients | 452 lookup terms
alias collisions: none


## 3. Query normalization (same logic as notebook 03)

In [6]:
def parse_query(query, lookup):
    """Raw pantry text (English, Tagalog, or Taglish) -> (canonical ingredient names, unmapped tokens)."""
    parts = re.split(r"[,;\n]|\bat\b|\band\b", query, flags=re.IGNORECASE)
    canonicals, unmapped = [], []
    for part in parts:
        part = part.strip()
        if not part:
            continue
        c, left = normalize_text(part, lookup)
        for x in c:
            if x not in canonicals:
                canonicals.append(x)
        unmapped.extend(left)
    return canonicals, unmapped

for q in ["may sitaw, bawang, at manok ako", "toyo, bawang, sibuyas, manok"]:
    print(q, "->", parse_query(q, lookup)[0])

may sitaw, bawang, at manok ako -> ['string beans', 'garlic', 'chicken']
toyo, bawang, sibuyas, manok -> ['soy sauce', 'garlic', 'onion', 'chicken']


## 4. Retrievers: overlap, BM25

In [7]:
def overlap_scores(query_canon):
    q = set(query_canon)
    out = np.zeros(len(df))
    if not q:
        return out
    for i, s in enumerate(df["canon_set"]):
        m = len(q & s)
        out[i] = m + 0.5 * (m / max(1, len(s)))   # matched count first, coverage breaks ties
    return out

bm25 = BM25Okapi([[x.replace(" ", "_") for x in lst] for lst in df["canon_list"]])

def bm25_scores(query_canon):
    toks = [x.replace(" ", "_") for x in query_canon]
    return bm25.get_scores(toks) if toks else np.zeros(len(df))

In [8]:
embedder = SentenceTransformer(EMBED_MODEL)

doc_text = (df["recipe_name"].astype(str) + ". Ingredients: " + df["canon_list"].map(lambda l: ", ".join(l))).tolist()

# Embeddings are cached. Delete the .npy file if you change the model or the recipe data.
cache = Path("data/processed") / f"recipe_vecs_{EMBED_MODEL.split('/')[-1]}.npy"
recipe_vecs = np.load(cache) if cache.exists() else None

if recipe_vecs is None or recipe_vecs.shape[0] != len(df):
    t0 = time.time()
    passages = ["passage: " + t for t in doc_text]       # e5 models expect this prefix
    chunks = []
    for start in range(0, len(passages), 200):
        chunks.append(embedder.encode(passages[start:start + 200], normalize_embeddings=True, show_progress_bar=False))
        print(f"embedded {min(start + 200, len(passages))}/{len(passages)} | {time.time() - t0:.0f}s", flush=True)
    recipe_vecs = np.vstack(chunks)
    np.save(cache, recipe_vecs)

print(recipe_vecs.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

(1983, 384)


In [9]:
def dense_scores(query, query_canon):
    q_text = ("Ingredients: " + ", ".join(query_canon)) if query_canon else query
    q_vec = embedder.encode(["query: " + q_text], normalize_embeddings=True)[0]
    return recipe_vecs @ q_vec

def rank_of(scores):
    order = np.argsort(-scores, kind="stable")
    ranks = np.empty(len(scores), dtype=int)
    ranks[order] = np.arange(1, len(scores) + 1)
    return ranks

def search(query, method="hybrid", k=TOP_K):
    """Return ([(row_position, score), ...], query_canonicals)."""
    qc, _ = parse_query(query, lookup)
    parts = {"overlap": overlap_scores(qc), "bm25": bm25_scores(qc), "dense": dense_scores(query, qc)}
    if method == "hybrid":
        scores = sum(1.0 / (RRF_K + rank_of(parts[p])) for p in HYBRID_PARTS)
    else:
        scores = parts[method]
    top = np.argsort(-scores, kind="stable")[:k]
    return [(int(i), float(scores[i])) for i in top], qc

# Compare the four methods on one query
query = "may sitaw, bawang, at manok ako"
for m in ["overlap", "bm25", "hybrid", "dense"]:
    hits, qc = search(query, m, k=5)
    print(f"{m:8s}", [df.iloc[i]["recipe_name"] for i, _ in hits])
print("normalized query:", qc)

overlap  ['Chicken Pochero Recipe (Pocherong Manok)', 'Ginataang Manok with Sitaw at Kalabasa', 'Chicken and Veggie Soup', 'Adobong Sitaw with Chicken', 'Chicken Kare Kare']
bm25     ['Chicken Pochero Recipe (Pocherong Manok)', 'Ginataang Manok with Sitaw at Kalabasa', 'Chicken and Veggie Soup', 'Adobong Sitaw with Chicken', 'Chicken Kare Kare']
hybrid   ['Chicken and Veggie Soup', 'Chicken Pochero', 'Sauteed Shrimp with String Beans', 'Chop Suey with Chicken', 'Ginataang Gulay']
dense    ['Chicken and Veggie Soup', 'Paprika Chicken Drumsticks', 'Garlic Fried Chicken', 'Beer-Marinated Grilled Chicken', 'Oven-Baked Barbecue Chicken']
normalized query: ['string beans', 'garlic', 'chicken']


In [10]:
name_key = df["recipe_name"].astype(str).str.strip().str.lower()
versions_per_dish = name_key.value_counts()

def top_dishes(query, method="hybrid", k=TOP_K, pool=50):
    """Top-k distinct dishes. If several rows share a name, the best-ranked version is kept."""
    hits, qc = search(query, method, k=pool) 
    seen, out = set(), []
    for i, s in hits:
        key = name_key.iloc[i]
        if key in seen:
            continue
        seen.add(key)
        out.append((i, s))
        if len(out) == k:
            break
    return out, qc

## 5. RAG prompt template

In [11]:
SYSTEM_PROMPT = (
    "You are a Filipino recipe assistant. "
    "Use ONLY the recipes provided in the context. "
    "Do not invent ingredients, quantities, cooking times, or steps that are not in the context. "
    "Respect the user's constraints (time, diet, cooking style) and never recommend a recipe that breaks them. "
    "Reply in the same language as the user's message (English, Tagalog, or Taglish)."
)

MATCH_LINE = (
    "Matched and missing ingredients, exactly as listed in the context"
    if GIVE_MATCH_HINTS else
    "Matched ingredients (the user has them) and missing ingredients"
)

USER_TEMPLATE = """The user has these ingredients: {pantry}
(Normalized ingredient names: {normalized})
User constraints: {constraints}

Context recipes (already ranked, best first):
{context}

Give exactly {n} recommendations, one for each recipe in the context, in the same order.
For each recipe, write:
- Why it fits: one sentence.
- """ + MATCH_LINE + """.
- Substitutes: for each missing ingredient, a reasonable substitute, or "none" if there is no good one.
- Steps: the cooking instructions as at most 5 short numbered steps. Keep all quantities and times."""

def format_context(hits, query_canon, max_chars=700):
    qset = set(query_canon)
    blocks = []
    for n, (i, _) in enumerate(hits, start=1):
        row = df.iloc[i]
        hint = ""
        if GIVE_MATCH_HINTS:
            have = [x for x in row["canon_list"] if x in qset]
            miss = [x for x in row["canon_list"] if x not in qset]
            hint = (f"\nMatched ingredients (the user has): {', '.join(have) or 'none'}"
                    f"\nMissing ingredients: {', '.join(miss) or 'none'}")
        time_txt = str(row.get("total_time", "")).strip() or "not stated"
        blocks.append(
            f"[Recipe {n}] {row['recipe_name']}\n"
            f"Total time: {time_txt}\n"
            f"Ingredients: {row['ingredients']}{hint}\n"
            f"Instructions: {str(row['instructions'])[:max_chars]}"
        )
    return "\n\n".join(blocks)

def build_messages(pantry, hits, query_canon, constraints="none"):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_TEMPLATE.format(
            pantry=pantry,
            normalized=", ".join(query_canon) or "none recognized",
            constraints=constraints,
            context=format_context(hits, query_canon),
            n=len(hits),
        )},
    ]

## 6. Generation through Ollama (prints the answer as it is written)

In [12]:
def generate(messages):
    out = ""
    for chunk in ollama.chat(
        model=LLM_MODEL,
        messages=messages,
        stream=True,
        options={"temperature": 0, "seed": SEED, "num_predict": 500},
    ):
        piece = chunk["message"]["content"]
        print(piece, end="", flush=True)
        out += piece
    return out

## 7. End-to-end test

In [15]:
query = "may tufo ako at isda at magic sarap"
METHOD = "hybrid"     # "overlap", "bm25", "dense", or "hybrid"

hits, qc = top_dishes(query, METHOD, k=TOP_K)
print("Normalized query:", qc)
print(f"Retrieved dishes ({METHOD}):")
for i, s in hits:
    n = int(versions_per_dish[name_key.iloc[i]])
    print(f"  {s:.4f} | {df.iloc[i]['recipe_name']}" + (f"  (+{n - 1} other versions)" if n > 1 else ""))

print("\n--- LLM answer ---\n")
answer = generate(build_messages(query, hits, qc))

Normalized query: ['maggi magic sarap']
Retrieved dishes (hybrid):
  0.0207 | Tortang Talong  (+2 other versions)
  0.0191 | Ginisang Repolyo with Eggs
  0.0190 | Bistek Silog

--- LLM answer ---

Here are my recommendations for each recipe:

**Recipe 1: Tortang Talong**

* Why it fits: This recipe fits because it's a popular Filipino dish that can be made with common ingredients.
* Matched and missing ingredients:
	+ Matched: maggi magic sarap
	+ Missing: eggplant, egg, salt, cooking oil
* Substitutes:
	+ Eggplant: none
	+ Egg: none
	+ Salt: none
	+ Cooking oil: vegetable oil
* Steps:
	1. Grill the eggplant until the color of skin turns almost black.
	2. Let the eggplant cool for a while then peel off the skin. Set aside.
	3. Crack the eggs and place in a bowl. Add salt and beat.
	4. Place the eggplant on a flat surface and flatten using a fork.
	5. Dip the flattened eggplant in the beaten egg mixture and fry in hot oil until golden brown.

**Recipe 2: Ginisang Repolyo with Eggs**

* 